In [0]:
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
score_model(pd.DataFrame({
  "messages": [
    [
      {
        "role": "user",
        "content": "Can i use wheel python package for deploying an agent as dependency in databricks for serving container?"
      }
    ]
  ]
}))

{'predictions': {'messages': [{'role': 'assistant',
    'content': 'Yes – a Python\u202fwheel can be used as a dependency for the code that runs inside a Databricks serving container (or any other Databricks “agent” such as a job or notebook).  \n\n### How it works\n\n1. **Build the wheel** in your CI pipeline (the docs show the `databricks bundle deploy` command that packages a wheel together with notebooks and other assets).  \n2. **Upload the wheel** to a location that the cluster can read – either:  \n   * Inside the **Databricks Asset Bundle** (the bundle is unpacked on the release agent and the wheel is copied to DBFS automatically), or  \n   * Directly to **DBFS** (e.g. `dbfs:/Volumes/<catalog>/<schema>/wheels/my_pkg‑1.0.0‑py3‑none‑any.whl`).  \n3. **Reference the wheel** when you create the serving endpoint (or any job that runs the model):  \n\n   ```python\n   from databricks import connect\n   from databricks.connect import DatabricksSession, DatabricksEnv\n\n   # Tell Datab

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS classic_stable_cykcbe_catalog.agentops.python_dependencies;

In [0]:
catalog_name = "classic_stable_cykcbe_catalog"
schema_name = "agentops"
volume_name = "python_dependencies"
wheel_file_name = "agentops_framework-1.0.0-py3-none-any.whl"


In [0]:
import os

volume_mount_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
wheel_path = os.path.join(volume_mount_path, wheel_file_name)

print("Expected wheel path:")
print(wheel_path)

print("\nListing contents of the volume:")
for f in os.listdir(volume_mount_path):
    print(" -", f)

# Check existence
if os.path.exists(wheel_path):
    print(f"\n✅ Found wheel file at: {wheel_path}")
else:
    raise FileNotFoundError(
        f"Wheel file not found at {wheel_path}. "
        f"Make sure you uploaded {wheel_file_name} to the volume."
    )


Expected wheel path:
/Volumes/classic_stable_cykcbe_catalog/agentops/python_dependencies/agentops_framework-1.0.0-py3-none-any.whl

Listing contents of the volume:
 - agentops_framework-1.0.0-py3-none-any.whl

✅ Found wheel file at: /Volumes/classic_stable_cykcbe_catalog/agentops/python_dependencies/agentops_framework-1.0.0-py3-none-any.whl


In [0]:
# Important:
# - This cell runs inside the container (or cluster) context.
# - The /Volumes/... path must be accessible to the container.
# - If using jobs, make sure the task's cluster or container has UC volumes enabled.

print(f"Installing wheel from: {wheel_path}")

# Use %pip magic for proper environment handling in Databricks notebooks
# Note: this must be a top-level cell, not inside a function.

Installing wheel from: /Volumes/classic_stable_cykcbe_catalog/agentops/python_dependencies/agentops_framework-1.0.0-py3-none-any.whl


In [0]:
# Databricks interpreter resets variables across %pip cells, so we keep this separate.
# Adjust extra options as needed (e.g., --force-reinstall during development).

install_cmd = f"pip install --upgrade {wheel_path}"
print(install_cmd)

pip install --upgrade /Volumes/classic_stable_cykcbe_catalog/agentops/python_dependencies/agentops_framework-1.0.0-py3-none-any.whl


In [0]:
# Actually execute the pip install
# This will restart the Python interpreter for the notebook after installation.
%pip install --upgrade "$wheel_path"

Processing /Volumes/classic_stable_cykcbe_catalog/agentops/python_dependencies/agentops_framework-1.0.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 28.9 MB/s eta 0:00:00
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-940d5aa7-2c91-4406-bbeb-1d583e641816
    Can't uninstall 'blinker'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, 

In [0]:
# Re-run Cell 2 (config) after the %pip cell, then run this one.

# Replace 'my_agent' with the actual package name inside the wheel
package_import_name = "framework"  # e.g., if your wheel defines a package named 'my_agent'

try:
    agent_pkg = __import__(package_import_name)
    print(f"✅ Successfully imported package '{package_import_name}'")
    print("Package location:", agent_pkg.__file__)
except ImportError as e:
    print(f"❌ Could not import package '{package_import_name}'")
    raise e

✅ Successfully imported package 'framework'
Package location: /local_disk0/.ephemeral_nfs/envs/pythonEnv-940d5aa7-2c91-4406-bbeb-1d583e641816/lib/python3.12/site-packages/framework/__init__.py


In [0]:
# This is just an example; adjust to match your package's API.
# Assume the wheel exposes something like: my_agent.Agent or my_agent.create_agent()

from framework.agent_base import AgentOPSBase   # adjust to your real API

# Example initialization

In [0]:
from framework.agent_base import AgentOPSBase
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse
import mlflow, uuid

class MyAgent(AgentOPSBase):
    def __init__(self):
        config = {
            "agent": {"name": "demo-agent", "version": "1.0.0"},
            "llm": {"endpoint": "databricks-gpt-oss-120b", "max_tokens": 256},
            "guardrails": {
                "enabled": False,
                "pre_llm": {},
                "post_llm": {},
            },
        }
        super().__init__(config)

    def _process_request(self, messages, context=None, custom_inputs=None):
        user_msg = messages[-1].content
        return ChatAgentResponse(
            messages=[ChatAgentMessage(
                id=str(uuid.uuid4()), role="assistant",
                content=f"Hello! You said: {user_msg}"
            )]
        )

agent = MyAgent()
response = agent.predict(
    messages=[ChatAgentMessage(role="user", content="Say hello from the agent")]
)
print(response.messages[0].content)

Hello! You said: Say hello from the agent


Trace(trace_id=tr-5f72ec5016ce3814995e9f0e0a6d3de4)

In [0]:
import os
os.chdir("/Workspace/Users/narendra.merla@databricks.com/agentops_demo/agent_development/agent")
import sys
sys.path.insert(0, "/Workspace/Users/narendra.merla@databricks.com/agentops_demo/agent_development/agent")

  # This single line does everything — loads config, inits guardrails, connects to VS + LLM
exec(open("/Workspace/Users/narendra.merla@databricks.com/agentops_demo/agent_development/agent/notebooks/app.py").read())

  # AGENT is now ready — same as the serving container
from mlflow.types.agent import ChatAgentMessage
response = AGENT.predict(messages=[ChatAgentMessage(role="user", content="What is Databricks sql warehouse ?")])
print(response.messages[0].content)

**Databricks SQL Warehouse** (formerly called *SQL endpoints*) is the compute layer that powers Databricks SQL.  

- It is a **dedicated SQL compute resource** that runs queries and dashboards against the tables stored in your lakehouse (Delta Lake files, Unity Catalog‑managed assets, etc.).  
- The warehouse is **decoupled from storage** – you can scale the compute independently of the data files.  
- It can be **provisioned as a serverless or provisioned warehouse**; Databricks recommends using the serverless option when it’s available.  
- When you run a query against a stopped warehouse, Databricks automatically starts it (if you have access) 【Source 1】.  
- SQL warehouses appear in the compute drop‑down of all workspace UIs that support SQL (query editor, Catalog Explorer, dashboards, notebooks, jobs, etc.) and can be sorted, searched, and managed from the **SQL Warehouses** sidebar view 【Source 1】.  
- They provide the underlying **ANSI‑SQL engine with Delta Lake extensions**, en

Trace(trace_id=tr-c0b2ef4378165fdf5c6520d43d10c361)